# 1 · LCEL Chains — Intake & Routing

The first two steps of the pipeline are **deterministic**, so we use plain **LCEL chains** (no agents):

1. **Intake** — turn a raw ticket into a structured object.
2. **Routing** — classify the ticket and decide which specialist should handle it.

Both use `llm.with_structured_output(...)` with a Pydantic schema.

In [1]:
%run aurora_common.py

C:\Users\akhawaja\git\cs4603\wk4_capstone\aurora_common.py:42: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## Intake chain
Extract structured fields from the raw ticket text.

In [2]:
from pydantic import BaseModel, Field
from typing import Optional


class Ticket(BaseModel):
    customer_name: Optional[str] = Field(None, description="Customer name if mentioned")
    order_id: Optional[int] = Field(None, description="Order number if mentioned")
    intent: str = Field(description="Short phrase describing what the customer wants")
    sentiment: str = Field(description="positive | neutral | negative")


intake_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the structured fields from the customer ticket."),
    ("human", "{ticket}"),
])

intake_chain = intake_prompt | llm_noreason.with_structured_output(Ticket)

sample = "Hi, this is Ada. My order 1001 still hasn't arrived and I'm frustrated. When will it ship?"
intake_chain.invoke({"ticket": sample})

Ticket(customer_name='Ada', order_id=1001, intent='Check shipping status', sentiment='negative')

## Routing chain
Classify category + priority and pick the specialist.

> **TODO (student):** add more categories, and a `needs_action` boolean for tickets that require a
> mutation (refund/cancel). Consider few-shot examples to improve accuracy.

In [3]:
from typing import Literal


class Routing(BaseModel):
    category: Literal["order_status", "returns_refunds", "product_policy", "technical", "other"]
    priority: Literal["low", "medium", "high"]
    route_to: Literal["knowledge_agent", "order_agent", "web_agent"] = Field(
        description="Which specialist should handle this ticket"
    )


routing_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the ticket and choose the best specialist to handle it."),
    ("human", "{ticket}"),
])

routing_chain = routing_prompt | llm_noreason.with_structured_output(Routing)
routing_chain.invoke({"ticket": sample})

Routing(category='order_status', priority='high', route_to='order_agent')

## Try a few tickets
> **TODO (student):** compose `intake_chain` and `routing_chain` into a single LCEL pipeline that
> returns both the `Ticket` and the `Routing` for one input.

In [4]:
tickets = [
    "What's the status of order 1001?",
    "How many days do I have to return opened electronics?",
    "Can you cancel order 1003? I ordered by mistake.",
]
for t in tickets:
    print(t)
    print("  ->", routing_chain.invoke({"ticket": t}))
    print()

What's the status of order 1001?


  -> category='order_status' priority='medium' route_to='order_agent'

How many days do I have to return opened electronics?


  -> category='product_policy' priority='low' route_to='knowledge_agent'

Can you cancel order 1003? I ordered by mistake.


  -> category='order_status' priority='medium' route_to='order_agent'



### Definition of done
- `intake_chain` returns a valid `Ticket` for varied inputs.
- `routing_chain` picks a sensible `route_to` for order, policy, and action tickets.
- Bonus: a combined chain returning `{ticket, routing}` in one call.